# Latent Potential Demand Estimation: Tri-Model Ensemble

In this notebook, we walk through the final ML architecture used to estimate the latent maximum monthly volume potential for January 2026. Because observed sales are constrained (stockouts, credit caps, cooler limits), this is a **right-censored demand problem**.

## The Tri-Model Meta Ensemble
1. **Model A (XGBoost AFT):** Uses survival analysis (`objective='survival:aft'`) to learn latent headroom by placing dynamically calibrated upper bounds on unconstrained outlets.
2. **Model B (LightGBM Quantile):** Learns the 95th percentile upper envelope (`alpha=0.95`), using probability-weighted training to down-weight censored records.
3. **Model C (Peer Benchmark):** Calculates historical 95th percentile capacity within precise feature-space macro-clusters.
4. **Meta-Blender & Guardrails:** A Ridge regressor blends the models, and strict business guardrails prevent extreme runaway extrapolations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

# Load the Final Gold model input
df = pd.read_parquet('../data/gold/model_input.parquet')
print(f'Loaded {len(df)} outlets with {df.shape[1]} engineered features.')
df[['Outlet_ID', 'total_volume', 'censor_probability', 'poi_gravity_score', 'cooler_efficiency_percentile']].head()

## 1. The Censor Probability Engine
We calculate a continuous `censor_probability` ∈ [0,1] using a hybrid heuristic + KMeans approach. This captures constraints like flat volume (plateau), high return ratios, and stockout proxies.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['censor_probability'], bins=40, color='crimson', kde=True)
plt.title('Distribution of Outlet Censor Probability')
plt.xlabel('Probability of being constrained (Censor Probability)')
plt.ylabel('Number of Outlets')
plt.axvline(0.65, color='black', linestyle='--', label='AFT Strict Bound Threshold (>0.65)')
plt.legend()
plt.show()

## 2. Meta-Ensemble Weights & Predictions
We trained the Tri-Model ensemble and blended it using Ridge Regression. Let's load the final generated predictions to see how the guardrails performed.

In [ ]:
preds = pd.read_csv('../output/insightai_final_predictions.csv')
merged = df.merge(preds, on='Outlet_ID')

print(f"Total Observed Historical Volume: {merged['total_volume'].sum():,.2f} L")
print(f"Total Projected Network Potential (Jan 2026): {merged['Maximum_Monthly_Liters'].sum():,.2f} L")

uplift = (merged['Maximum_Monthly_Liters'].sum() / merged['total_volume'].sum()) - 1
print(f"Network-wide Demand Uplift (Uncapped): {uplift*100:.1f}%")

### Guardrails in Action
The business guardrails ensure our predictions don't break the laws of physics. The prediction must be $\ge$ historical robust stable floor, and $\le$ 1.5x the extreme peer benchmark.

In [ ]:
# Let's visualize the shift from observed historical to the model's true latent potential
plt.figure(figsize=(10, 5))
sns.kdeplot(merged['total_volume'].clip(upper=400), label='Observed Historical Volume', fill=True, color='grey')
sns.kdeplot(merged['Maximum_Monthly_Liters'].clip(upper=400), label='Predicted Latent Potential (Ensemble)', fill=True, color='blue')
plt.title('Demand Distribution: Observed vs. Tri-Model Latent Potential')
plt.xlabel('Volume (Liters) - Clipped at 400L for visualization')
plt.legend()
plt.show()